# 03 - Trays, Modules, And Event Selection

IceTray gets its name from the `I3Tray`: a tray is a chain of modules. Frames enter the tray, each module gets a chance to inspect or change the frame, and some modules can decide whether to keep or drop frames.

This notebook keeps the code visible. We will write a few small Python functions and then add them to a tray.


In [ ]:
from pathlib import Path
import os

from icecube import icetray, dataclasses, dataio
from I3Tray import I3Tray

DATA_GCD = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_0101_78_503_GCD.i3.zst')
DATA_FILE = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_Subrun00000000_00000000.i3.zst')

print('GCD exists: ', DATA_GCD.exists())
print('Data exists:', DATA_FILE.exists())
print('Importing dataio above is important: it registers I3Reader and I3Writer with IceTray.')


## A function that counts hit DOMs

A module can be an ordinary Python function. IceTray will call the function once per frame.

The function below looks for a pulse map, counts how many DOMs have pulses, and writes that number back into the frame as `TutorialHitDOMs`.


In [ ]:
def find_pulse_key(frame):
    possible_keys = ['SplitInIcePulses', 'SplitInIceDSTPulses', 'SRTInIcePulses', 'InIcePulses', 'OfflinePulses']
    for key in possible_keys:
        if key in frame:
            return key
    return None

def add_hit_dom_count(frame):
    pulse_key = find_pulse_key(frame)
    if pulse_key is None:
        # Returning True means: keep this frame moving through the tray.
        return True

    pulse_map = dataclasses.I3RecoPulseSeriesMap.from_frame(frame, pulse_key)
    hit_doms = 0
    for omkey, pulses_on_one_dom in pulse_map:
        if len(pulses_on_one_dom) > 0:
            hit_doms += 1

    frame['TutorialHitDOMs'] = icetray.I3Int(hit_doms)
    return True

print('Defined add_hit_dom_count. It adds a new key called TutorialHitDOMs.')


## A function that selects events

If a tray function returns `False`, IceTray drops that frame from the rest of the tray. That is a simple way to make event selections.


In [ ]:
def keep_events_with_many_doms(frame, minimum_doms=20):
    if 'TutorialHitDOMs' not in frame:
        return False

    hit_doms = frame['TutorialHitDOMs'].value
    return hit_doms >= minimum_doms

print('Defined a selection function: keep frames with at least minimum_doms hit DOMs.')


## A small QFilterMask helper

`QFilterMask` stores named filter decisions. A filter counts as passed when both `condition_passed` and `prescale_passed` are true. We will use this later as an optional selection.


In [ ]:
def filter_passed(frame, filter_name):
    if 'QFilterMask' not in frame:
        return False
    if filter_name not in frame['QFilterMask']:
        return False

    result = frame['QFilterMask'][filter_name]
    return result.condition_passed and result.prescale_passed

def keep_muon_filter(frame):
    # If this filter name is not present in your file, inspect QFilterMask and change it.
    return filter_passed(frame, 'MuonFilter_13')

print('Defined filter_passed and keep_muon_filter.')


## Choose an output location

The selected `.i3.zst` file can be large, so we write it under `/data/user/<username>/IceTray_tutorial/` instead of inside the git repository.


In [ ]:
username = os.environ.get('USER', 'icecube_user')
output_dir = Path(f'/data/user/{username}/IceTray_tutorial')
output_dir.mkdir(parents=True, exist_ok=True)
selected_file = output_dir / 'selected_min20doms.i3.zst'

print('Selected events will be written to:')
print(selected_file)


## Build and run the tray

Read this cell from top to bottom. The tray reads frames, counts hit DOMs in Physics frames, drops events with fewer than 20 hit DOMs, and writes the frames that remain.

We only execute 200 frames while learning. For a real run, you would increase this or remove the limit.


In [ ]:
tray = I3Tray()

tray.AddModule('I3Reader', 'reader', FilenameList=[str(DATA_GCD), str(DATA_FILE)])

tray.AddModule(add_hit_dom_count, 'add_hit_dom_count', Streams=[icetray.I3Frame.Physics])
tray.AddModule(keep_events_with_many_doms, 'minimum_dom_selection', minimum_doms=20, Streams=[icetray.I3Frame.Physics])

tray.AddModule(
    'I3Writer',
    'writer',
    Filename=str(selected_file),
    Streams=[icetray.I3Frame.TrayInfo, icetray.I3Frame.DAQ, icetray.I3Frame.Physics],
)

print('Starting the tray now...')
tray.Execute(200)
tray.Finish()
print('Finished. Output written to:', selected_file)


## Practice

1. Change `minimum_doms=20` to `minimum_doms=5`. Does the output file get larger?
2. Add `keep_muon_filter` to the tray after `add_hit_dom_count`.
3. Add a print statement inside `add_hit_dom_count` for the first few events. Then remove it once it gets annoying.
4. Write a new function that keeps events with total charge above a threshold.
